In [1]:
!conda env list


# conda environments:
#
# * -> active
# + -> frozen
                         /home/njm12/ATMS_523/envs/era5-850
                         /home/njm12/ATMS_523/envs/xarray-climate
base                 *   /opt/conda



In [2]:
!conda run -p /home/njm12/ATMS_523/envs/xarray-climate python -m ipykernel install --user --name xarray-climate --display-name "Python (xarray-climate)"

Installed kernelspec xarray-climate in /home/njm12/.local/share/jupyter/kernels/xarray-climate



In [3]:
import sys
print(sys.executable)

/home/njm12/ATMS_523/envs/xarray-climate/bin/python


In [4]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# -----------------------------------------------------------
# USER INPUTS
# -----------------------------------------------------------
lat = 40.361667    # Coordinates are for Warsaw, IL (the approximate center of the region of interest)
lon = -91.420556   # West longitude is negative
timezone = -6      # CST (UTC-6). Use -5 if you want CDT instead.
start_year = 1991
end_year = 2020
zenith = 90.833    # Official sunrise/sunset zenith angle (deg)

# -----------------------------------------------------------
# NOAA Sunrise/Sunset Function
# -----------------------------------------------------------
def calculate_sun_time(date, latitude, longitude, zenith, timezone, is_sunrise):
    
    N = date.timetuple().tm_yday
    lngHour = longitude / 15
    
    if is_sunrise:
        t = N + ((6 - lngHour) / 24)
    else:
        t = N + ((18 - lngHour) / 24)

    M = (0.9856 * t) - 3.289

    L = M + (1.916 * np.sin(np.radians(M))) + \
        (0.020 * np.sin(np.radians(2 * M))) + 282.634
    L = L % 360

    RA = np.degrees(np.arctan(0.91764 * np.tan(np.radians(L))))
    RA = RA % 360

    Lquadrant  = (np.floor(L / 90)) * 90
    RAquadrant = (np.floor(RA / 90)) * 90
    RA = RA + (Lquadrant - RAquadrant)
    RA = RA / 15

    sinDec = 0.39782 * np.sin(np.radians(L))
    cosDec = np.cos(np.arcsin(sinDec))

    cosH = (np.cos(np.radians(zenith)) - (sinDec * np.sin(np.radians(latitude)))) / \
           (cosDec * np.cos(np.radians(latitude)))

    if cosH > 1:
        return None
    if cosH < -1:
        return None

    if is_sunrise:
        H = 360 - np.degrees(np.arccos(cosH))
    else:
        H = np.degrees(np.arccos(cosH))

    H = H / 15

    T = H + RA - (0.06571 * t) - 6.622

    UT = (T - lngHour) % 24

    localT = (UT + timezone) % 24

    return localT


# -----------------------------------------------------------
# Build 1991–2020 Daily Dataset
# -----------------------------------------------------------
dates = pd.date_range(f"{start_year}-01-01",
                      f"{end_year}-12-31", freq="D")

data = []

for date in dates:
    sunrise = calculate_sun_time(date, lat, lon, zenith, timezone, True)
    sunset = calculate_sun_time(date, lat, lon, zenith, timezone, False)
    
    data.append([date, sunrise, sunset])

df = pd.DataFrame(data, columns=["date", "sunrise_hour", "sunset_hour"])


# -----------------------------------------------------------
# Convert decimal hours to seconds for averaging
# -----------------------------------------------------------
df["sunrise_sec"] = df["sunrise_hour"] * 3600
df["sunset_sec"] = df["sunset_hour"] * 3600


# -----------------------------------------------------------
# Assign Meteorological Seasons
# -----------------------------------------------------------
def get_season(month):
    if month in [12, 1, 2]:
        return "DJF"
    elif month in [3, 4, 5]:
        return "MAM"
    elif month in [6, 7, 8]:
        return "JJA"
    else:
        return "SON"

# Assign meteorological seasons
df["season"] = df["date"].dt.month.apply(get_season)

# Force correct seasonal order
season_order = ["DJF", "MAM", "JJA", "SON"]
df["season"] = pd.Categorical(df["season"],
                               categories=season_order,
                               ordered=True)

# -----------------------------------------------------------
# Compute Seasonal Averages
# -----------------------------------------------------------
seasonal = df.groupby("season").agg(
    sunrise_mean_sec=("sunrise_sec", "mean"),
    sunset_mean_sec=("sunset_sec", "mean"),
    n_days=("date", "count")
).reset_index()


# -----------------------------------------------------------
# Convert back to HH:MM:SS
# -----------------------------------------------------------
def sec_to_hms(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

seasonal["mean_sunrise"] = seasonal["sunrise_mean_sec"].apply(sec_to_hms)
seasonal["mean_sunset"] = seasonal["sunset_mean_sec"].apply(sec_to_hms)

print(seasonal[["season", "mean_sunrise", "mean_sunset", "n_days"]], '\n')

# -----------------------------------------------------------
# Compute Diurnal and Nocturnal Hours from Seasonal Means
# -----------------------------------------------------------

# Daylight duration (seconds)
seasonal["diurnal_sec"] = seasonal["sunset_mean_sec"] - seasonal["sunrise_mean_sec"]

# Nighttime duration (seconds)
seasonal["nocturnal_sec"] = 86400 - seasonal["diurnal_sec"] #86400 represents total number of seconds in one day (24 * 3600) 

# Convert seconds to hours
seasonal["diurnal_hours"] = seasonal["diurnal_sec"] / 3600
seasonal["nocturnal_hours"] = seasonal["nocturnal_sec"] / 3600

# -----------------------------------------------------------
# Output
# -----------------------------------------------------------

print("\nSeasonal Sunrise/Sunset Normals (1991–2020, CST)")
print("--------------------------------------------------")

for _, row in seasonal.iterrows():
    print(f"\n{row['season']}")
    print(f"  Mean Sunrise : {row['mean_sunrise']}")
    print(f"  Mean Sunset  : {row['mean_sunset']}")
    print(f"  Diurnal Hours: {row['diurnal_hours']:.2f}")
    print(f"  Nocturnal Hours: {row['nocturnal_hours']:.2f}")

  season mean_sunrise mean_sunset  n_days
0    DJF     07:15:36    17:08:22    2708
1    MAM     05:30:44    18:44:50    2760
2    JJA     04:54:43    19:23:12    2760
3    SON     06:19:00    17:29:08    2730 


Seasonal Sunrise/Sunset Normals (1991–2020, CST)
--------------------------------------------------

DJF
  Mean Sunrise : 07:15:36
  Mean Sunset  : 17:08:22
  Diurnal Hours: 9.88
  Nocturnal Hours: 14.12

MAM
  Mean Sunrise : 05:30:44
  Mean Sunset  : 18:44:50
  Diurnal Hours: 13.23
  Nocturnal Hours: 10.77

JJA
  Mean Sunrise : 04:54:43
  Mean Sunset  : 19:23:12
  Diurnal Hours: 14.47
  Nocturnal Hours: 9.53

SON
  Mean Sunrise : 06:19:00
  Mean Sunset  : 17:29:08
  Diurnal Hours: 11.17
  Nocturnal Hours: 12.83


/tmp/ipykernel_822/3483660777.py:118: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  seasonal = df.groupby("season").agg(
